# Лабораторная работа: стохастический градиентный бустинг

Сравнение собственной реализации (`sgb.py`) с `sklearn.ensemble.GradientBoostingClassifier` на датасете **Breast Cancer Wisconsin**. Оценка качества — стратифицированная k-fold кросс-валидация; время обучения замеряется отдельно на полной выборке при одинаковых гиперпараметрах.

In [1]:
import time

import numpy as np
import pandas as pd
from sklearn.base import clone
from sklearn.datasets import load_breast_cancer
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.preprocessing import StandardScaler

from sgb import StochasticGradientBoostingClassifier

RANDOM_STATE = 42
N_SPLITS = 5
SUBSAMPLE = 0.7
N_ESTIMATORS = 100
LEARNING_RATE = 0.1
MAX_DEPTH = 3

data = load_breast_cancer()
X, y = data.data, data.target
feature_names = data.feature_names
target_names = data.target_names

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

print("Объектов:", X.shape[0], "признаков:", X.shape[1])
print("Классы:", target_names, "распределение:", np.bincount(y))

Объектов: 569 признаков: 30
Классы: ['malignant' 'benign'] распределение: [212 357]


## Кросс-валидация (точность)

In [ ]:
cv = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=RANDOM_STATE)

custom_clf = StochasticGradientBoostingClassifier(
    n_estimators=N_ESTIMATORS,
    learning_rate=LEARNING_RATE,
    max_depth=MAX_DEPTH,
    subsample=SUBSAMPLE,
    random_state=RANDOM_STATE,
)

sk_clf = GradientBoostingClassifier(
    n_estimators=N_ESTIMATORS,
    learning_rate=LEARNING_RATE,
    max_depth=MAX_DEPTH,
    subsample=SUBSAMPLE,
    random_state=RANDOM_STATE,
)

cv_custom = cross_val_score(
    custom_clf, X_scaled, y, cv=cv, scoring="accuracy", n_jobs=-1
)
cv_sklearn = cross_val_score(
    sk_clf, X_scaled, y, cv=cv, scoring="accuracy", n_jobs=-1
)

results_acc = pd.DataFrame(
    {
        "Модель": ["Своя SGB", "sklearn GradientBoosting"],
        "Средняя accuracy": [cv_custom.mean(), cv_sklearn.mean()],
        "Std": [cv_custom.std(), cv_sklearn.std()],
    }
)
results_acc

## Время обучения на полной выборке

In [ ]:
def bench_fit(estimator, X, y, n_runs=5):
    times = []
    for _ in range(n_runs):
        est = clone(estimator)
        t0 = time.perf_counter()
        est.fit(X, y)
        times.append(time.perf_counter() - t0)
    return float(np.mean(times)), float(np.std(times))


t_custom_mean, t_custom_std = bench_fit(custom_clf, X_scaled, y)
t_sk_mean, t_sk_std = bench_fit(sk_clf, X_scaled, y)

results_time = pd.DataFrame(
    {
        "Модель": ["Своя SGB", "sklearn GradientBoosting"],
        "Время обучения (с), среднее": [t_custom_mean, t_sk_mean],
        "Std": [t_custom_std, t_sk_std],
    }
)
results_time

## Сводная таблица

In [ ]:
summary = pd.DataFrame(
    {
        "Модель": ["Своя SGB", "sklearn GradientBoosting"],
        "CV accuracy (mean)": [cv_custom.mean(), cv_sklearn.mean()],
        "CV accuracy (std)": [cv_custom.std(), cv_sklearn.std()],
        "Время fit (с)": [t_custom_mean, t_sk_mean],
    }
)
summary

Параметры эксперимента: `subsample=0.7` (стохастическое обучение на ~70% объектов на каждом дереве), `n_estimators=100`, `learning_rate=0.1`, `max_depth=3`, данные масштабированы `StandardScaler`. Численные значения в ячейках выше используются в `README.md` после запуска ноутбука.